# Stemming and Lemmatization

The bookshop catalog stores documents word by word. A user searches "car repair"
and misses titles like "cars", "repairing cars", and "repaired vehicles" because
each surface form is a distinct token in the index. Tokenization fixed splitting;
it did not fix the fact that one root word has many written shapes. This demo
closes that gap: first it decides which tokens are worth indexing at all, then it
collapses inflected forms onto a common stem or lemma, and finally it resolves the
bookshop query from the chapter opener.

**Learning goals:**
- See why stop-word removal is a heuristic, not a rule, and when it must not run
- Read the Porter measure `m` and the five-step structure behind the black box
- Compare Porter, Lancaster, and Snowball aggressiveness on the same words
- Contrast rule-based stemming with dictionary lemmatization (went to go, better to good)
- Watch Snowball and spaCy diverge on German and French
- Measure stemming closing the recall gap on a BM25-style match

**Prerequisites:** Section 3.1 (tokenization and normalization)

In [1]:
import re
import nltk
import spacy
from nltk.stem import PorterStemmer, LancasterStemmer, SnowballStemmer, WordNetLemmatizer
from shared.display import print_table, display_md
from shared.text import tokenize, stopwords_for

In [2]:
# One-time data/model downloads (quiet, safe to re-run)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

porter = PorterStemmer()
lancaster = LancasterStemmer()
snowball_en = SnowballStemmer("english")
snowball_de = SnowballStemmer("german")
snowball_fr = SnowballStemmer("french")
wn = WordNetLemmatizer()

nlp_en = spacy.load("en_core_web_sm")
nlp_de = spacy.load("de_core_news_sm")
nlp_fr = spacy.load("fr_core_news_sm")

## 1. Stop words: what to keep out

A handful of very frequent words dominate any text: "the", "of", "and", "to", "is"
together make up roughly a quarter of tokens in typical prose, yet carry almost no
content signal. A stop-word list is the standard shortcut: a fixed set of function
words to discard before indexing.

In [3]:
display_md(
    f"**Stop-word list sizes (nltk):**\n\n"
    f"| Language | Words |\n|---|---|\n"
    f"| English | {len(stopwords_for('en'))} |\n"
    f"| German | {len(stopwords_for('de'))} |\n"
    f"| French | {len(stopwords_for('fr'))} |\n\n"
    f"Sample English: {sorted(stopwords_for('en'))[:12]}"
)

**Stop-word list sizes (nltk):**

| Language | Words |
|---|---|
| English | 198 |
| German | 232 |
| French | 157 |

Sample English: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any']

Dropping stop words works most of the time and fails visibly the rest of the time,
in the small class of queries where the stop word is content-bearing.

In [4]:
en_stops = stopwords_for("en")
cases = [
    ("It", "Stephen King novel; the title is the stop word"),
    ("IT security", "IT = information technology, not the pronoun"),
    ("To be or not to be", "the whole line is stop words"),
    ("The Who", "band name, both tokens on the list"),
]
rows = []
for phrase, note in cases:
    kept = [t for t in tokenize(phrase) if t not in en_stops]
    rows.append([phrase, " ".join(kept) if kept else "(empty)", note])
print_table(rows, headers=["Query", "After stop-word removal", "Why it breaks"])

| Query              | After stop-word removal   | Why it breaks                                  |
|:-------------------|:--------------------------|:-----------------------------------------------|
| It                 | (empty)                   | Stephen King novel; the title is the stop word |
| IT security        | security                  | IT = information technology, not the pronoun   |
| To be or not to be | (empty)                   | the whole line is stop words                   |
| The Who            | (empty)                   | band name, both tokens on the list             |

BM25 already handles frequent words gracefully: inverse document frequency drives corpus-wide common terms toward zero weight, and term-frequency saturation stops any single term from dominating. That makes aggressive removal less necessary. The modern default is to **keep stop words in the index** (storage is cheap) but skip them when generating phrase collocations, computing term similarities, or building compact classifier features.

**Caution: never strip stop words from a phrase query.** Dropping them before phrase
matching turns "The Who" into an empty query and "not guilty" into "guilty". A
phrase-aware pipeline must leave those tokens in place.

## 2. Stemming: the Porter measure

Once stop words are handled, we still have "car", "cars", "carrying", "carried",
"carrier": one concept, five tokens. Stemming applies ordered rules that strip
suffixes so variants of one root collapse to the same token. Porter (1980) is the
classical English stemmer.

Porter writes every word as `[C](VC)^m[V]`, where `C` is a run of consonants, `V` a
run of vowels, and the exponent `m` is the word's **measure**. Rules that would
over-stem are gated behind a minimum `m`, so "ATE" is stripped from "OPERATE"
(`m` high) but not from "MATE" (`m` low).

In [5]:
def measure(word: str):
    """Porter's measure m and the collapsed consonant/vowel pattern."""
    cv = []
    for i, ch in enumerate(word.lower()):
        if ch in "aeiou":
            cv.append("V")
        elif ch == "y":
            cv.append("V" if cv and cv[-1] == "C" else "C")
        else:
            cv.append("C")
    collapsed = "".join(c for i, c in enumerate(cv) if i == 0 or c != cv[i - 1])
    return collapsed.count("VC"), collapsed

print_table(
    [[w, patt, m] for w in ["tree", "trouble", "troubles", "oats"]
     for m, patt in [measure(w)]],
    headers=["Word", "Pattern", "Measure m"],
)

| Word     | Pattern   |   Measure m |
|:---------|:----------|------------:|
| tree     | CV        |           0 |
| trouble  | CVCV      |           1 |
| troubles | CVCVC     |           2 |
| oats     | VC        |           1 |

The algorithm runs five ordered steps, roughly 60 suffix rules in total. Within each
step only the first matching rule fires.

| Step | Goal | Example rules | Example |
| --- | --- | --- | --- |
| 1a | Strip plurals | SSES to SS, IES to I, S to nothing | ponies to poni, cats to cat |
| 1b | Strip past / progressive | (m>0) EED to EE, (v) ED, (v) ING | plastered to plaster, motoring to motor |
| 2 | Simplify derivational suffixes | (m>0) ATIONAL to ATE | relational to relate |
| 3 | Continue simplification | (m>0) ICATE to IC, ALIZE to AL | formalize to formal |
| 4 | Remove residual suffixes | (m>1) ION, ISM, ENT | adoption to adopt |
| 5 | Clean up trailing letters | (m>1) E to nothing, double to single | controll to control |

In [6]:
family = ["car", "cars", "carrying", "carried", "carrier"]
print_table([[w, porter.stem(w)] for w in family], headers=["Word", "Porter stem"])

| Word     | Porter stem   |
|:---------|:--------------|
| car      | car           |
| cars     | car           |
| carrying | carri         |
| carried  | carri         |
| carrier  | carrier       |

"carrying" and "carried" both collapse to `carri` (not to `carry`), and "carrier" is left intact. The stems are not pretty English, but three variants of the verb "carry" now share one token, which is all the index needs.

## 3. Lancaster and Snowball

Lancaster (Paice, 1990) is more aggressive: it strips suffixes iteratively until no
rule fires, producing shorter stems and more collisions.

In [7]:
print_table(
    [[w, lancaster.stem(w)] for w in ["one", "only", "organization", "organize"]],
    headers=["Word", "Lancaster stem"],
)

| Word         | Lancaster stem   |
|:-------------|:-----------------|
| one          | on               |
| only         | on               |
| organization | org              |
| organize     | org              |

"one" and "only" both stem to `on`, and "organization" and "organize" both to `org`. A document about "the only book on Rome" now collides with one about "book number one". Fine when recall dominates, harmful for a name or a precise query.

**Caution: Lancaster over-stems short words.** Use it when speed dominates and the
collection is large enough that a few token collisions do not move the ranking.
Prefer Porter or Snowball on short-document collections where each collision matters.

Snowball is Porter's own multi-language framework, covering about 20 European
languages. On English it is a lightly revised Porter; on German, French, and the
rest it is often the only rule-based option.

In [8]:
words = ["running", "universal", "university", "argue", "arguing", "argued",
         "happiness", "connection", "connected"]
print_table(
    [[w, porter.stem(w), snowball_en.stem(w), lancaster.stem(w)] for w in words],
    headers=["Word", "Porter", "Snowball", "Lancaster"],
)

| Word       | Porter   | Snowball   | Lancaster   |
|:-----------|:---------|:-----------|:------------|
| running    | run      | run        | run         |
| universal  | univers  | univers    | univers     |
| university | univers  | univers    | univers     |
| argue      | argu     | argu       | argu        |
| arguing    | argu     | argu       | argu        |
| argued     | argu     | argu       | argu        |
| happiness  | happi    | happi      | happy       |
| connection | connect  | connect    | connect     |
| connected  | connect  | connect    | connect     |

In [9]:
de_words = ["Wagen", "Wägen", "Reparatur", "reparieren"]
print_table([[w, snowball_de.stem(w)] for w in de_words], headers=["German word", "Snowball (de)"])
display_md(
    "The German stemmer folds the umlaut (`Wägen` and `Wagen` both to `wag`) but only "
    "approximates the stem: the noun \"Reparatur\" and the verb \"reparieren\" share a root, "
    "yet no pure suffix rule maps them to the same token."
)

| German word   | Snowball (de)   |
|:--------------|:----------------|
| Wagen         | wag             |
| Wägen         | wag             |
| Reparatur     | reparatur       |
| reparieren    | repari          |

The German stemmer folds the umlaut (`Wägen` and `Wagen` both to `wag`) but only approximates the stem: the noun "Reparatur" and the verb "reparieren" share a root, yet no pure suffix rule maps them to the same token.

## 4. Lemmatization: dictionary-based reduction

Stemming is fast and needs no data, but it produces linguistically wrong stems and
cannot handle strong inflections: "go" and "went" share no letters, so no suffix
rule maps them together. Lemmatization instead looks the word up in a dictionary and
returns its recorded base form, using three ingredients: a rule set for regular
inflections, an exception list for irregular forms (went to go, mice to mouse), and
a dictionary to validate candidates. It needs the word's part of speech.

In [10]:
examples = [("carried", "v"), ("carrier", "n"), ("mice", "n"), ("went", "v"), ("better", "a")]
print_table(
    [[w, pos, wn.lemmatize(w, pos)] for w, pos in examples],
    headers=["Word", "POS", "WordNet lemma"],
)

| Word    | POS   | WordNet lemma   |
|:--------|:------|:----------------|
| carried | v     | carry           |
| carrier | n     | carrier         |
| mice    | n     | mouse           |
| went    | v     | go              |
| better  | a     | good            |

"went" lemmatizes to `go` and "better" (adjective) to `good`. No rule-based stemmer can do either: the mapping lives in the exception list, not in the spelling.

spaCy runs a full pipeline and tags part of speech automatically, so the lemmatizer
applies without the caller passing a tag.

In [11]:
sentence = "She carried the mice yesterday and went home"
print_table(
    [[t.text, t.pos_, t.lemma_] for t in nlp_en(sentence)],
    headers=["Token", "POS", "spaCy lemma"],
)

| Token     | POS   | spaCy lemma   |
|:----------|:------|:--------------|
| She       | PRON  | she           |
| carried   | VERB  | carry         |
| the       | DET   | the           |
| mice      | NOUN  | mouse         |
| yesterday | NOUN  | yesterday     |
| and       | CCONJ | and           |
| went      | VERB  | go            |
| home      | NOUN  | home          |

## 5. Multi-lingual comparison

Every stemmer assumes one language; the language detector from the previous section
picks it. Snowball is the rule-based option outside English, spaCy the lemmatizer.
On the same sentence they diverge on inflected verbs.

In [12]:
def compare(sentence, stemmer, nlp):
    toks = sentence.split()
    return [[t, stemmer.stem(t), lem.lemma_] for t, lem in zip(toks, nlp(sentence))]

print_table(
    compare("Nous avons aperçu les châteaux", snowball_fr, nlp_fr),
    headers=["French token", "Snowball", "spaCy lemma"],
)

| French token   | Snowball   | spaCy lemma   |
|:---------------|:-----------|:--------------|
| Nous           | nous       | nous          |
| avons          | avon       | avoir         |
| aperçu         | aperçu     | apercevoir    |
| les            | le         | le            |
| châteaux       | château    | château       |

In [13]:
print_table(
    compare("die Häuser und ihre Häuschen", snowball_de, nlp_de),
    headers=["German token", "Snowball", "spaCy lemma"],
)

| German token   | Snowball   | spaCy lemma   |
|:---------------|:-----------|:--------------|
| die            | die        | der           |
| Häuser         | haus       | Haus          |
| und            | und        | und           |
| ihre           | ihr        | ihr           |
| Häuschen       | hausch     | Häuschen      |

In French, Snowball reduces "avons" to `avon` by suffix stripping, while spaCy maps it to the infinitive `avoir` and "aperçu" to `apercevoir`; both keep the accents and reduce plural `châteaux` to `château`. In German, Snowball over-stems the diminutive "Häuschen" to `hausch` (a non-word matching nothing), while spaCy keeps `Häuschen` as its own lemma. Neither splits compounds: "Wolkenkratzer" stays whole, which is the next section's concern.

## 6. The payoff: closing the bookshop gap

Back to the opener. Index three short documents and query "car repair", first with
raw tokens, then with Porter stems.

In [14]:
docs = {
    "d1": "We repaired the cars yesterday.",
    "d2": "Repairing a car is expensive.",
    "d3": "Vehicle maintenance and service.",
}
query = "car repair"

def match(query, doc, stem=False):
    q = set(tokenize(query))
    d = set(tokenize(doc))
    if stem:
        q = {porter.stem(t) for t in q}
        d = {porter.stem(t) for t in d}
    return len(q & d), sorted(q & d)

rows = []
for did, doc in docs.items():
    raw_n, raw_terms = match(query, doc)
    stem_n, stem_terms = match(query, doc, stem=True)
    rows.append([did, doc, f"{raw_n}: {raw_terms}", f"{stem_n}: {stem_terms}"])
print_table(rows, headers=["Doc", "Text", "Raw overlap", "Stemmed overlap"])

| Doc   | Text                             | Raw overlap   | Stemmed overlap      |
|:------|:---------------------------------|:--------------|:---------------------|
| d1    | We repaired the cars yesterday.  | 0: []         | 2: ['car', 'repair'] |
| d2    | Repairing a car is expensive.    | 1: ['car']    | 2: ['car', 'repair'] |
| d3    | Vehicle maintenance and service. | 0: []         | 0: []                |

Without stemming, `d1` matches nothing (it has "cars" and "repaired", not "car" and "repair") and `d2` matches only "car". After Porter stemming both match the full query. `d3` still matches nothing: "vehicle" is a synonym, not a spelling variant, and stemming cannot bridge that gap. Closing it needs the semantic methods of later chapters.

## Summary

| Method | Approach | Strength | Weakness |
| --- | --- | --- | --- |
| Stop-word removal | Drop frequent function words | Smaller index, less noise | Destroys phrases and negation |
| Porter | Ordered suffix rules, measure-gated | Solid recall baseline | Some over/under-stemming |
| Lancaster | Aggressive iterative stripping | Maximum conflation, fast | Frequent collisions |
| Snowball | Revised Porter, ~20 languages | Best general and multilingual | Still rule-based |
| WordNet lemma | Dictionary lookup with POS | Correct base forms, irregulars | Needs POS, English |
| spaCy lemma | Full pipeline, auto POS | Handles irregulars, many languages | Model loading, weak on strong verbs |

<div style="border-left: 4px solid #C8102E; background: rgba(200, 16, 46, 0.06); padding: 0.6em 0.9em; margin: 0.6em 0; border-radius: 4px;">
<strong style="color:#C8102E; text-transform:uppercase; font-size:0.78em; letter-spacing:0.06em;">Takeaway</strong><br>
Stemming is a handful of string operations that boost recall at some precision cost; lemmatization is heavier but returns real base forms and handles irregulars. Both still sit on the query path of every serious lexical search stack, and hybrid retrieval pairs them with dense embeddings rather than replacing them.
</div>

## Try it yourself

1. Find a word where Porter and Snowball produce different stems, and explain why.
2. Lemmatize "saw" as a noun and as a verb. Why do the two disagree?
3. Add a fourth document to Section 6 that only matches after lemmatization, not
   after stemming.

In [15]:
word = "saw"
display_md(
    f"**'{word}' under each method:**\n\n"
    f"- WordNet noun: {wn.lemmatize(word, 'n')}\n"
    f"- WordNet verb: {wn.lemmatize(word, 'v')}\n"
    f"- Porter stem: {porter.stem(word)}"
)

**'saw' under each method:**

- WordNet noun: saw
- WordNet verb: saw
- Porter stem: saw